# Week 7 – Outlier Detection and Data Quality

**Objective:** Apply outlier detection to the Residential MLS dataset using the Interquartile Range (IQR) method.

This notebook focuses on the **Residential-filtered sold dataset** because the official Week 7 requirement applies IQR filtering to:

- `ClosePrice`
- `LivingArea`
- `DaysOnMarket`

The workflow preserves the full dataset by adding outlier flags first, then creates a separate clean filtered dataset for analysis.


## 1. Import Packages and Define File Paths


In [33]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path("/Users/amyliu/Desktop/IDX")

# Candidate Week 6 input paths
sold_candidates = [
    BASE_DIR / "data" / "generated" / "week6" / "sold_week6_engineered_metrics.csv"
]

listing_candidates = [
    BASE_DIR / "data" / "generated" / "week6" / "listing_week6_engineered_metrics.csv"
]

SOLD_FILE = next((path for path in sold_candidates if path.exists()), None)
LISTINGS_FILE = next((path for path in listing_candidates if path.exists()), None)

OUTPUT_DIR = BASE_DIR / "data" / "generated" / "week7"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if SOLD_FILE is None:
    raise FileNotFoundError(
        "Could not find sold_week6_engineered_metrics.csv in data/week6 or data/generated/week6."
    )

print("Sold input file:", SOLD_FILE)
print("Listing input file:", LISTINGS_FILE if LISTINGS_FILE is not None else "Not found / optional")
print("Output folder:", OUTPUT_DIR)

Sold input file: /Users/amyliu/Desktop/IDX/data/generated/week6/sold_week6_engineered_metrics.csv
Listing input file: /Users/amyliu/Desktop/IDX/data/generated/week6/listing_week6_engineered_metrics.csv
Output folder: /Users/amyliu/Desktop/IDX/data/generated/week7


## 2. Load Week 6 Sold Dataset


In [34]:
sold = pd.read_csv(SOLD_FILE, low_memory=False)

print(f"Sold rows loaded: {len(sold):,}")
print(f"Sold columns loaded: {sold.shape[1]:,}")

sold.head()

Sold rows loaded: 397,603
Sold columns loaded: 93


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,CloseDate,ClosePrice,ListAgentFirstName,...,implausible_coord_flag,price_ratio,close_to_original_list_ratio,price_per_sqft,days_on_market,close_year,close_month,yrmo,listing_to_contract_days,contract_to_close_days
0,Mlslistings,Mlslistings,"Carpet,Tile,Wood",True,False,499000.0,551985747,2024-01-26,240000.0,Joan,...,False,0.480962,0.480962,210.526316,777,2024,1,2024-01,777.0,65.0
1,SanDiego,SanDiego,NaN,False,False,759900.0,522107581,2024-01-05,815000.0,Michael,...,False,1.072510,1.072510,412.867275,33,2024,1,2024-01,114.0,919.0
2,SanDiego,SanDiego,NaN,False,False,739900.0,510919001,2024-01-05,810000.0,Michael,...,False,1.094743,1.094743,410.334347,228,2024,1,2024-01,255.0,778.0
3,Mlslistings,Mlslistings,NaN,False,NaN,NaN,1079166779,2024-01-30,858000.0,David,...,False,NaN,NaN,430.075188,0,2024,1,2024-01,188.0,-188.0
4,Southland,Southland,NaN,False,False,1890500.0,1075037759,2024-01-29,1890500.0,Karen,...,False,1.000000,1.000000,591.891046,0,2024,1,2024-01,0.0,0.0


## 3. Confirm and Apply Residential Filter

The analysis should focus on **Residential** records only. This prevents non-residential property types, such as land, leases, or commercial properties, from distorting the IQR thresholds.


In [36]:
print("PropertyType distribution before Residential filter:")
print(sold["PropertyType"].value_counts(dropna=False))

before_residential_filter = len(sold)

sold = sold[sold["PropertyType"] == "Residential"].copy()

after_residential_filter = len(sold)

print(f"\nRows before Residential filter: {before_residential_filter:,}")
print(f"Rows after Residential filter: {after_residential_filter:,}")
print(f"Rows removed by Residential filter: {before_residential_filter - after_residential_filter:,}")

print("\nPropertyType distribution after Residential filter:")
print(sold["PropertyType"].value_counts(dropna=False))

PropertyType distribution before Residential filter:
PropertyType
Residential    397603
Name: count, dtype: int64

Rows before Residential filter: 397,603
Rows after Residential filter: 397,603
Rows removed by Residential filter: 0

PropertyType distribution after Residential filter:
PropertyType
Residential    397603
Name: count, dtype: int64


## 4. Check Key Numeric Fields

The Week 7 deliverable requires outlier detection for:

- `ClosePrice`
- `LivingArea`
- `DaysOnMarket`


In [37]:
outlier_fields = [
    "ClosePrice",
    "LivingArea",
    "DaysOnMarket"
]

field_check = pd.DataFrame({
    "field": outlier_fields,
    "exists": [col in sold.columns for col in outlier_fields]
})

field_check

,field,exists
0,ClosePrice,True
1,LivingArea,True
2,DaysOnMarket,True


## 5. Convert Numeric Fields and Review Initial Distributions

The IQR calculation requires numeric values. This section converts the fields and reviews their pre-filtering distribution.


In [38]:
for col in outlier_fields:
    sold[col] = pd.to_numeric(sold[col], errors="coerce")

before_numeric_summary = sold[outlier_fields].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

before_numeric_summary

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
ClosePrice,397601.0,1.185616e+06,5.922380e+06,0.0,203000.0,340000.0,575000.0,820000.0,1300000.0,2850000.0,5550000.0,989500000.0
LivingArea,397374.0,1.904351e+03,2.701781e+04,0.0,604.0,839.0,1247.0,1641.0,2217.0,3558.0,5280.0,17021321.0
DaysOnMarket,397603.0,3.733679e+01,5.353925e+01,-288.0,0.0,1.0,8.0,19.0,48.0,131.0,229.0,12430.0


## 6. Apply Business-Rule Data Quality Flags

Before applying IQR, obvious invalid values are flagged. These records are not deleted from the full dataset.

Business rules used:

- `ClosePrice <= 0` is invalid
- `LivingArea <= 0` is invalid
- `DaysOnMarket < 0` is invalid


In [39]:
sold["invalid_close_price_flag"] = sold["ClosePrice"].isna() | (sold["ClosePrice"] <= 0)
sold["invalid_living_area_flag"] = sold["LivingArea"].isna() | (sold["LivingArea"] <= 0)
sold["invalid_days_on_market_flag"] = sold["DaysOnMarket"].isna() | (sold["DaysOnMarket"] < 0)

business_rule_summary = pd.DataFrame({
    "flag": [
        "invalid_close_price_flag",
        "invalid_living_area_flag",
        "invalid_days_on_market_flag"
    ],
    "flagged_records": [
        sold["invalid_close_price_flag"].sum(),
        sold["invalid_living_area_flag"].sum(),
        sold["invalid_days_on_market_flag"].sum()
    ]
})

business_rule_summary["flagged_pct"] = business_rule_summary["flagged_records"] / len(sold)

business_rule_summary

,flag,flagged_records,flagged_pct
0,invalid_close_price_flag,3,0.000008
1,invalid_living_area_flag,373,0.000938
2,invalid_days_on_market_flag,46,0.000116


## 7. Calculate IQR Thresholds

For each numeric field, the IQR rule defines outliers as values below:

`Q1 - 1.5 × IQR`

or above:

`Q3 + 1.5 × IQR`


In [40]:
iqr_thresholds = []

for col in outlier_fields:
    valid_series = sold[col].dropna()

    q1 = valid_series.quantile(0.25)
    q3 = valid_series.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    iqr_thresholds.append({
        "field": col,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound
    })

iqr_thresholds_df = pd.DataFrame(iqr_thresholds)

iqr_thresholds_df

,field,q1,q3,iqr,lower_bound,upper_bound
0,ClosePrice,575000.0,1300000.0,725000.0,-512500.0,2387500.0
1,LivingArea,1247.0,2217.0,970.0,-208.0,3672.0
2,DaysOnMarket,8.0,48.0,40.0,-52.0,108.0


## 8. Add IQR Outlier Flags

Instead of removing rows immediately, this section creates one IQR flag for each numeric field.


In [41]:
for _, row in iqr_thresholds_df.iterrows():
    col = row["field"]
    lower = row["lower_bound"]
    upper = row["upper_bound"]

    flag_col = f"{col.lower()}_iqr_outlier_flag"

    sold[flag_col] = (
        sold[col].notna() &
        ((sold[col] < lower) | (sold[col] > upper))
    )

iqr_flag_cols = [
    "closeprice_iqr_outlier_flag",
    "livingarea_iqr_outlier_flag",
    "daysonmarket_iqr_outlier_flag"
]

iqr_flag_summary = sold[iqr_flag_cols].sum().reset_index()
iqr_flag_summary.columns = ["flag", "flagged_records"]
iqr_flag_summary["flagged_pct"] = iqr_flag_summary["flagged_records"] / len(sold)

iqr_flag_summary

,flag,flagged_records,flagged_pct
0,closeprice_iqr_outlier_flag,29410,0.073968
1,livingarea_iqr_outlier_flag,17540,0.044114
2,daysonmarket_iqr_outlier_flag,30269,0.076129


## 9. Create Combined Removal Flag

The clean analysis dataset excludes rows that fail any business rule or trigger any IQR outlier flag. The full flagged dataset remains preserved.


In [42]:
sold["any_business_rule_invalid_flag"] = (
    sold["invalid_close_price_flag"] |
    sold["invalid_living_area_flag"] |
    sold["invalid_days_on_market_flag"]
)

sold["any_iqr_outlier_flag"] = (
    sold["closeprice_iqr_outlier_flag"] |
    sold["livingarea_iqr_outlier_flag"] |
    sold["daysonmarket_iqr_outlier_flag"]
)

sold["remove_from_clean_analysis_flag"] = (
    sold["any_business_rule_invalid_flag"] |
    sold["any_iqr_outlier_flag"]
)

overall_flag_summary = pd.DataFrame({
    "category": [
        "Business-rule invalid records",
        "IQR outlier records",
        "Total records removed from clean analysis"
    ],
    "record_count": [
        sold["any_business_rule_invalid_flag"].sum(),
        sold["any_iqr_outlier_flag"].sum(),
        sold["remove_from_clean_analysis_flag"].sum()
    ]
})

overall_flag_summary["record_pct"] = overall_flag_summary["record_count"] / len(sold)

overall_flag_summary

,category,record_count,record_pct
0,Business-rule invalid records,422,0.001061
1,IQR outlier records,62132,0.156266
2,Total records removed from clean analysis,62349,0.156812


## 10. Create Clean Filtered Analysis Dataset

The full dataset keeps all records and flags. The clean analysis dataset removes flagged records for downstream analysis.


In [43]:
sold_clean_filtered = sold[~sold["remove_from_clean_analysis_flag"]].copy()

print(f"Full flagged dataset rows: {len(sold):,}")
print(f"Clean filtered dataset rows: {len(sold_clean_filtered):,}")
print(f"Rows removed: {len(sold) - len(sold_clean_filtered):,}")
print(f"Removal percentage: {(len(sold) - len(sold_clean_filtered)) / len(sold):.2%}")

Full flagged dataset rows: 397,603
Clean filtered dataset rows: 335,254
Rows removed: 62,349
Removal percentage: 15.68%


## 11. Compare Before and After Filtering

This table compares row counts, medians, and means before and after filtering.


In [44]:
comparison_rows = []

for col in outlier_fields:
    comparison_rows.append({
        "field": col,
        "before_row_count": sold[col].notna().sum(),
        "after_row_count": sold_clean_filtered[col].notna().sum(),
        "before_median": sold[col].median(),
        "after_median": sold_clean_filtered[col].median(),
        "median_change": sold_clean_filtered[col].median() - sold[col].median(),
        "before_mean": sold[col].mean(),
        "after_mean": sold_clean_filtered[col].mean(),
        "mean_change": sold_clean_filtered[col].mean() - sold[col].mean()
    })

before_after_comparison = pd.DataFrame(comparison_rows)

before_after_comparison

,field,before_row_count,after_row_count,before_median,after_median,median_change,before_mean,after_mean,mean_change
0,ClosePrice,397601,335254,820000.0,785000.0,-35000.0,1.185616e+06,898084.150535,-287532.209591
1,LivingArea,397374,335254,1641.0,1568.0,-73.0,1.904351e+03,1673.697779,-230.653678
2,DaysOnMarket,397603,335254,19.0,16.0,-3.0,3.733679e+01,26.375196,-10.961592


## 12. Percentile Review Before and After Filtering

Percentiles help confirm that extreme tails were reduced while the typical market range was preserved.


In [45]:
before_percentiles = sold[outlier_fields].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

after_percentiles = sold_clean_filtered[outlier_fields].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

before_percentiles["dataset"] = "before_filtering"
after_percentiles["dataset"] = "after_filtering"

percentile_comparison = pd.concat([
    before_percentiles,
    after_percentiles
]).reset_index().rename(columns={"index": "field"})

percentile_comparison

,field,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,dataset
0,ClosePrice,397601.0,1.185616e+06,5.922380e+06,0.00,203000.0,340000.0,575000.0,820000.0,1300000.0,2850000.0,5550000.0,989500000.0,before_filtering
1,LivingArea,397374.0,1.904351e+03,2.701781e+04,0.00,604.0,839.0,1247.0,1641.0,2217.0,3558.0,5280.0,17021321.0,before_filtering
2,DaysOnMarket,397603.0,3.733679e+01,5.353925e+01,-288.00,0.0,1.0,8.0,19.0,48.0,131.0,229.0,12430.0,before_filtering
3,ClosePrice,335254.0,8.980842e+05,4.599077e+05,1.15,210000.0,340000.0,563770.0,785000.0,1150000.0,1850000.0,2250000.0,2387500.0,after_filtering
4,LivingArea,335254.0,1.673698e+03,6.268312e+02,1.00,605.0,828.0,1212.0,1568.0,2035.0,2908.0,3408.0,3672.0,after_filtering
5,DaysOnMarket,335254.0,2.637520e+01,2.558334e+01,0.00,0.0,1.0,7.0,16.0,39.0,83.0,102.0,108.0,after_filtering


## 13. Listing-Side Outlier Check

The official Week 7 deliverable focuses on sold-side fields, especially `ClosePrice`.  
As an optional check, this section applies similar logic to the Residential listing dataset using:

- `ListPrice`
- `LivingArea`
- `DaysOnMarket`

This supports later Tableau analysis for new listings and listing-side market activity.


In [46]:
if LISTINGS_FILE is not None:
    listings = pd.read_csv(LISTINGS_FILE, low_memory=False)

    print(f"Listing rows loaded: {len(listings):,}")
    print("PropertyType distribution before Residential filter:")
    print(listings["PropertyType"].value_counts(dropna=False))

    listings = listings[listings["PropertyType"] == "Residential"].copy()

    print(f"\nListing rows after Residential filter: {len(listings):,}")
    print("PropertyType distribution after Residential filter:")
    print(listings["PropertyType"].value_counts(dropna=False))
else:
    listings = None
    print("Listing file not found. Optional listing-side check skipped.")

Listing rows loaded: 540,183
PropertyType distribution before Residential filter:
PropertyType
Residential    540183
Name: count, dtype: int64

Listing rows after Residential filter: 540,183
PropertyType distribution after Residential filter:
PropertyType
Residential    540183
Name: count, dtype: int64


In [47]:
if listings is not None:
    listing_outlier_fields = [
        "ListPrice",
        "LivingArea",
        "DaysOnMarket"
    ]

    for col in listing_outlier_fields:
        listings[col] = pd.to_numeric(listings[col], errors="coerce")

    listings["invalid_list_price_flag"] = listings["ListPrice"].isna() | (listings["ListPrice"] <= 0)
    listings["invalid_living_area_flag"] = listings["LivingArea"].isna() | (listings["LivingArea"] <= 0)
    listings["invalid_days_on_market_flag"] = listings["DaysOnMarket"].isna() | (listings["DaysOnMarket"] < 0)

    listing_iqr_thresholds = []

    for col in listing_outlier_fields:
        valid_series = listings[col].dropna()

        q1 = valid_series.quantile(0.25)
        q3 = valid_series.quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        listing_iqr_thresholds.append({
            "field": col,
            "q1": q1,
            "q3": q3,
            "iqr": iqr,
            "lower_bound": lower_bound,
            "upper_bound": upper_bound
        })

    listing_iqr_thresholds_df = pd.DataFrame(listing_iqr_thresholds)

    for _, row in listing_iqr_thresholds_df.iterrows():
        col = row["field"]
        lower = row["lower_bound"]
        upper = row["upper_bound"]

        flag_col = f"{col.lower()}_iqr_outlier_flag"

        listings[flag_col] = (
            listings[col].notna() &
            ((listings[col] < lower) | (listings[col] > upper))
        )

    listings["any_listing_business_rule_invalid_flag"] = (
        listings["invalid_list_price_flag"] |
        listings["invalid_living_area_flag"] |
        listings["invalid_days_on_market_flag"]
    )

    listings["any_listing_iqr_outlier_flag"] = (
        listings["listprice_iqr_outlier_flag"] |
        listings["livingarea_iqr_outlier_flag"] |
        listings["daysonmarket_iqr_outlier_flag"]
    )

    listings["remove_from_listing_clean_analysis_flag"] = (
        listings["any_listing_business_rule_invalid_flag"] |
        listings["any_listing_iqr_outlier_flag"]
    )

    listings_clean_filtered = listings[~listings["remove_from_listing_clean_analysis_flag"]].copy()

    listing_before_after_comparison = pd.DataFrame([
        {
            "field": col,
            "before_row_count": listings[col].notna().sum(),
            "after_row_count": listings_clean_filtered[col].notna().sum(),
            "before_median": listings[col].median(),
            "after_median": listings_clean_filtered[col].median(),
            "median_change": listings_clean_filtered[col].median() - listings[col].median(),
            "before_mean": listings[col].mean(),
            "after_mean": listings_clean_filtered[col].mean(),
            "mean_change": listings_clean_filtered[col].mean() - listings[col].mean()
        }
        for col in listing_outlier_fields
    ])

    listing_before_after_comparison
else:
    listing_iqr_thresholds_df = pd.DataFrame()
    listings_clean_filtered = pd.DataFrame()
    listing_before_after_comparison = pd.DataFrame()

In [50]:

    print("Listing-Side Outlier Check:")

    print(f"Original listing rows: {len(listings):,}")
    print(f"Clean filtered listing rows: {len(listings_clean_filtered):,}")
    print(f"Rows removed: {len(listings) - len(listings_clean_filtered):,}")
    print(f"Removal percentage: {(len(listings) - len(listings_clean_filtered)) / len(listings):.2%}")

    print("\nBusiness-rule invalid flags:")
    print(
        listings[
            [
                "invalid_list_price_flag",
                "invalid_living_area_flag",
                "invalid_days_on_market_flag"
            ]
        ].sum()
    )

    print("\nIQR outlier flags:")
    print(
        listings[
            [
                "listprice_iqr_outlier_flag",
                "livingarea_iqr_outlier_flag",
                "daysonmarket_iqr_outlier_flag"
            ]
        ].sum()
    )

    print("\nCombined flags:")
    print(
        listings[
            [
                "any_listing_business_rule_invalid_flag",
                "any_listing_iqr_outlier_flag",
                "remove_from_listing_clean_analysis_flag"
            ]
        ].sum()
    )

    print("\nListing IQR thresholds:")
    display(listing_iqr_thresholds_df)

    print("\nListing before/after comparison:")
    display(listing_before_after_comparison)

Listing-Side Outlier Check:
Original listing rows: 540,183
Clean filtered listing rows: 446,571
Rows removed: 93,612
Removal percentage: 17.33%

Business-rule invalid flags:
invalid_list_price_flag          0
invalid_living_area_flag       915
invalid_days_on_market_flag     29
dtype: int64

IQR outlier flags:
listprice_iqr_outlier_flag       45490
livingarea_iqr_outlier_flag      26718
daysonmarket_iqr_outlier_flag    45168
dtype: int64

Combined flags:
any_listing_business_rule_invalid_flag       944
any_listing_iqr_outlier_flag               93146
remove_from_listing_clean_analysis_flag    93612
dtype: int64

Listing IQR thresholds:


,field,q1,q3,iqr,lower_bound,upper_bound
0,ListPrice,580000.0,1375000.0,795000.0,-612500.0,2567500.0
1,LivingArea,1247.0,2300.0,1053.0,-332.5,3879.5
2,DaysOnMarket,5.0,23.0,18.0,-22.0,50.0



Listing before/after comparison:


,field,before_row_count,after_row_count,before_median,after_median,median_change,before_mean,after_mean,mean_change
0,ListPrice,540183,446571,840000.0,795000.0,-45000.0,1.312997e+06,919969.445760,-393027.331630
1,LivingArea,539627,446571,1669.0,1583.0,-86.0,1.980059e+03,1699.666332,-280.392368
2,DaysOnMarket,540183,446571,11.0,10.0,-1.0,1.953990e+01,12.975128,-6.564776


## 14. Save Week 7 Outputs

The required outputs are:

- Full flagged sold dataset
- Clean filtered sold dataset
- IQR thresholds
- Business-rule summary
- Before/after comparison

Optional listing-side outputs are saved only if the listing file is found.


In [48]:
# Required sold-side outputs
full_flagged_output = OUTPUT_DIR / "sold_week7_full_flagged_dataset.csv"
clean_filtered_output = OUTPUT_DIR / "sold_week7_clean_filtered_dataset.csv"
iqr_thresholds_output = OUTPUT_DIR / "week7_iqr_thresholds.csv"
business_rule_output = OUTPUT_DIR / "week7_business_rule_summary.csv"
iqr_flag_output = OUTPUT_DIR / "week7_iqr_flag_summary.csv"
overall_flag_output = OUTPUT_DIR / "week7_overall_flag_summary.csv"
before_after_output = OUTPUT_DIR / "week7_before_after_comparison.csv"
percentile_comparison_output = OUTPUT_DIR / "week7_percentile_comparison.csv"

sold.to_csv(full_flagged_output, index=False)
sold_clean_filtered.to_csv(clean_filtered_output, index=False)
iqr_thresholds_df.to_csv(iqr_thresholds_output, index=False)
business_rule_summary.to_csv(business_rule_output, index=False)
iqr_flag_summary.to_csv(iqr_flag_output, index=False)
overall_flag_summary.to_csv(overall_flag_output, index=False)
before_after_comparison.to_csv(before_after_output, index=False)
percentile_comparison.to_csv(percentile_comparison_output, index=False)

print("Saved required Week 7 sold-side outputs:")
print(full_flagged_output)
print(clean_filtered_output)
print(iqr_thresholds_output)
print(business_rule_output)
print(iqr_flag_output)
print(overall_flag_output)
print(before_after_output)
print(percentile_comparison_output)

# Optional listing-side outputs
if listings is not None:
    listing_full_flagged_output = OUTPUT_DIR / "listing_week7_full_flagged_dataset.csv"
    listing_clean_filtered_output = OUTPUT_DIR / "listing_week7_clean_filtered_dataset.csv"
    listing_iqr_thresholds_output = OUTPUT_DIR / "week7_listing_iqr_thresholds.csv"
    listing_before_after_output = OUTPUT_DIR / "week7_listing_before_after_comparison.csv"

    listings.to_csv(listing_full_flagged_output, index=False)
    listings_clean_filtered.to_csv(listing_clean_filtered_output, index=False)
    listing_iqr_thresholds_df.to_csv(listing_iqr_thresholds_output, index=False)
    listing_before_after_comparison.to_csv(listing_before_after_output, index=False)

    print("\nSaved optional listing-side outputs:")
    print(listing_full_flagged_output)
    print(listing_clean_filtered_output)
    print(listing_iqr_thresholds_output)
    print(listing_before_after_output)

print("\nWeek 7 outlier detection completed successfully.")

Saved required Week 7 sold-side outputs:
/Users/amyliu/Desktop/IDX/data/generated/week7/sold_week7_full_flagged_dataset.csv
/Users/amyliu/Desktop/IDX/data/generated/week7/sold_week7_clean_filtered_dataset.csv
/Users/amyliu/Desktop/IDX/data/generated/week7/week7_iqr_thresholds.csv
/Users/amyliu/Desktop/IDX/data/generated/week7/week7_business_rule_summary.csv
/Users/amyliu/Desktop/IDX/data/generated/week7/week7_iqr_flag_summary.csv
/Users/amyliu/Desktop/IDX/data/generated/week7/week7_overall_flag_summary.csv
/Users/amyliu/Desktop/IDX/data/generated/week7/week7_before_after_comparison.csv
/Users/amyliu/Desktop/IDX/data/generated/week7/week7_percentile_comparison.csv

Saved optional listing-side outputs:
/Users/amyliu/Desktop/IDX/data/generated/week7/listing_week7_full_flagged_dataset.csv
/Users/amyliu/Desktop/IDX/data/generated/week7/listing_week7_clean_filtered_dataset.csv
/Users/amyliu/Desktop/IDX/data/generated/week7/week7_listing_iqr_thresholds.csv
/Users/amyliu/Desktop/IDX/data/gener

## Week 7 Summary

This notebook applied outlier detection to the Residential-filtered sold dataset. I first created business-rule flags for clearly invalid values, then calculated IQR thresholds for `ClosePrice`, `LivingArea`, and `DaysOnMarket`. I added separate IQR outlier flags and created a combined removal flag.

The full flagged dataset is preserved, and a separate clean filtered dataset is created for analysis. I also compared dataset size, medians, means, and percentiles before and after filtering to document the impact of the outlier treatment.

An optional listing-side check is included using `ListPrice`, `LivingArea`, and `DaysOnMarket` to support future listing-side dashboard analysis.
